## Data Collection

In [ ]:
!pip install statsmodels seaborn matplotlib scikit-learn scipy xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sbn
import os
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
# import kagglehub
# kagglehub.login()
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from scipy.stats import f_oneway
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor, HuberRegressor, PassiveAggressiveRegressor, RANSACRegressor, TheilSenRegressor, RANSACRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score, max_error, median_absolute_error, mean_poisson_deviance, mean_gamma_deviance, mean_tweedie_deviance,mutual_info_score
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import VotingRegressor, StackingRegressor

In [ ]:
train_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")


## Data Inspection

In [ ]:
train_df.shape

In [ ]:
train_df.describe(include='all').T


In [ ]:
train_df.info()

In [ ]:
train_df.head(10)

## Missing Value Analysis

In [ ]:
cols_with_missing = train_df.columns[train_df.isnull().sum() > 0]

report = pd.DataFrame({
    "ColumnName": cols_with_missing,
    "MissingValues": train_df[cols_with_missing].isnull().sum().values,
    "MissingPercentage": round(train_df[cols_with_missing].isnull().mean() * 100, 2),
    "Dtype": train_df[cols_with_missing].dtypes.values,
    "Values": [train_df[col].value_counts(dropna=True).to_dict() for col in cols_with_missing]
})

report = report.sort_values(by="MissingValues", ascending=False).reset_index(drop=True)

report

In [ ]:
# Calculate missing values
missing_data = train_df.isnull().sum().sort_values(ascending=False)
missing_percentage = (missing_data / len(train_df)) * 100

# Create a DataFrame to display missing data information
missing_info = pd.DataFrame({
    'Missing Values': missing_data,
    'Missing Percentage': missing_percentage,
    'Dtype': train_df.dtypes
})

# Filter columns with missing data
missing_info = missing_info[missing_info['Missing Values'] > 0]
# print(missing_info)
#

# Define target variable
target = 'SalePrice'

# Results list
results = []

# Analyze impact of missing values on SalePrice
for col in missing_info.index:
    missing_indicator = train_df[col].isnull().astype(int)
    mean_missing = train_df.loc[missing_indicator == 1, target].mean()
    mean_present = train_df.loc[missing_indicator == 0, target].mean()
    results.append({
        'Feature': col,
        'Missing %': missing_percentage[col],
        'Mean SalePrice (Missing)': mean_missing,
        'Mean SalePrice (Present)': mean_present,
        'Difference': mean_present - mean_missing
    })

# Create a DataFrame for the results
impact_df = pd.DataFrame(results).sort_values(by='Difference', ascending=False)
impact_df


In [ ]:
missing_info_sorted = missing_info.sort_values(by='Missing Percentage', ascending=True)

plt.figure(figsize=(10,8))
sbn.barplot(x='Missing Percentage', y=missing_info_sorted.index, data=missing_info_sorted, palette='magma')
plt.xlabel('Missing Percentage (%)')
plt.ylabel('Feature')
plt.title('Missing Data Percentage by Feature')
plt.show()


In [ ]:
# Plot the impact of missing values on SalePrice
plt.figure(figsize=(12, 8))
sbn.barplot(x='Difference', y='Feature', data=impact_df, palette='coolwarm')
plt.title('Impact of Missing Values on SalePrice')
plt.xlabel('Difference in Mean SalePrice (Present - Missing)')
plt.ylabel('Feature')
plt.show()

**The above insights do not validate or confirm the missing value analysis.**

### 5. Decide Strategy

* **>80% missing** → drop column (low signal).
* **<5–10% missing** → impute (median/mean for numeric, mode for categorical).
* **Domain-driven missingness** → treat as a category (e.g., `GarageType=NA` means *no garage*).
* **Flagging** → add indicator column to capture predictive signal of missingness.

In [ ]:
!pip install missingno

In [ ]:
import missingno as msno
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
msno.bar(train_df)
plt.show()

In [ ]:
missing_data = train_df.isnull().sum().reset_index()
missing_data = missing_data.rename(columns={"index":"columns",0:"count"})
missing_data['dtype'] = train_df.dtypes.values
missing_data.groupby("dtype").count()

In [ ]:
missing_data['values'] = [train_df[col].value_counts(dropna=True).to_dict() for col in train_df.columns.to_list()]

In [ ]:
missing_data.sort_values(by='count',ascending=False).T

In [ ]:
summary = train_df.isnull().sum().sort_values(ascending=False)
summary = summary[summary >0]
report = pd.DataFrame({
    "MissingValues": summary,
    "Dtype": train_df[summary.index].dtypes
})

# Add value_counts as dictionary for each column
report["ValueCounts"] = report.index.map(
    lambda col: train_df[col].value_counts().to_dict()
)

report


In [ ]:
df_before_clean = train_df.copy()

In [ ]:
sbn.histplot(df_before_clean['LotFrontage'],kde=True)

## Data Cleaning Pipeline

In [ ]:
category_columns = df_before_clean.select_dtypes(include=['object']).columns.to_list()
num_columns = df_before_clean.select_dtypes(include=['int64','float64']).columns.to_list()
high_cardinal_cat_columns = [col for col in category_columns if df_before_clean[col].nunique() > 10]
low_cardinal_cat_columns = [col for col in category_columns if df_before_clean[col].nunique() <= 10]

In [ ]:
text_cols = [col for col in df_before_clean.columns
             if df_before_clean[col].dtype == 'object' and df_before_clean[col].dropna().apply(lambda x: isinstance(x, str) and len(x.split()) > 2).any()]

print(text_cols)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np


class SkewedImputerMixin:
    def _numeric_imputer_strategy(self, df, skew_threshold=0.5):
        """Determine imputation strategy based on skewness for numerical columns"""
        # Exclude Id column if present
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

        self.numeric_strategies = {}
        for col in numeric_cols:
            # Handle columns with all NaN values
            if df[col].notna().sum() == 0:
                self.numeric_strategies[col] = "median"
                continue

            skewness = df[col].skew()
            # Use median for skewed data, mean for normal distribution
            strategy = "median" if abs(skewness) > skew_threshold else "mean"
            self.numeric_strategies[col] = strategy
        return numeric_cols


class MissingValueAnalyzerMixin:
    def analyze(self, df):
        """Analyze missing values and determine handling strategy"""
        self.data = df.copy()
        cols_with_missing = df.columns[df.isnull().sum() > 0]

        if len(cols_with_missing) == 0:
            print("No missing values found in the dataset.")
            self.sparse_columns = []
            return pd.DataFrame()

        report = pd.DataFrame(
            {
                "ColumnName": cols_with_missing,
                "MissingValues": df[cols_with_missing].isnull().sum().values,
                "MissingPercentage": round(
                    df[cols_with_missing].isnull().mean() * 100, 2
                ),
                "Dtype": df[cols_with_missing].dtypes.values,
                "UniqueValues": [
                    df[col].nunique(dropna=True) for col in cols_with_missing
                ],
            }
        ).sort_values(by="MissingPercentage", ascending=False)

        # Decide strategy based on missing percentage
        report["Strategy"] = report["MissingPercentage"].apply(
            lambda pct: "Drop Column" if pct > 80 else "Impute"
        )
        print("SUMMMARY")
        print(report)
        # Store columns to be dropped
        self.sparse_columns.extend(
            report.loc[report["Strategy"] == "Drop Column", "ColumnName"].tolist()
        )

        print(f"Found {len(cols_with_missing)} columns with missing values.")
        print(f"Columns to drop (>80% missing): {len(self.sparse_columns)}")

        return report


class ImputerPipelineMixin:
    def build_imputer_pipeline(self, df, cardinality_threshold=10, skew_threshold=0.5):
        """Build preprocessing pipeline with appropriate imputers for different column types"""
        # Drop sparse columns (>80% missing)
        df_clean = df.drop(columns=getattr(self, "sparse_columns", []), errors="ignore")
        print(f"Dropped {len(getattr(self, 'sparse_columns', []))} sparse columns")

        # Get numeric columns and their strategies
        numeric_cols = self._numeric_imputer_strategy(df_clean, skew_threshold)
        numeric_strategies = getattr(self, "numeric_strategies", {})

        # Get categorical columns and split by cardinality
        cat_cols = df_clean.select_dtypes(exclude=[np.number]).columns.tolist()
        low_card_cols = [
            col
            for col in cat_cols
            if df_clean[col].nunique(dropna=False) <= cardinality_threshold
        ]
        high_card_cols = [
            col
            for col in cat_cols
            if df_clean[col].nunique(dropna=False) > cardinality_threshold
        ]

        print(f"Numeric columns: {len(numeric_cols)}")
        print(f"Low cardinality categorical: {len(low_card_cols)}")
        print(f"High cardinality categorical: {len(high_card_cols)}")

        # Build transformers list
        transformers = []

        # Numeric transformer with skew-aware imputation
        if numeric_cols:

            def skewed_imputer(X):
                """Custom imputer that handles skewed numeric data"""
                if len(X.shape) == 1:
                    X = X.reshape(-1, 1)

                X_df = pd.DataFrame(X, columns=numeric_cols[: X.shape[1]])

                for i, col in enumerate(numeric_cols[: X.shape[1]]):
                    strategy = numeric_strategies.get(col, "median")
                    imputer = SimpleImputer(strategy=strategy)
                    X_df.iloc[:, [i]] = imputer.fit_transform(X_df.iloc[:, [i]])

                return X_df.values

            numeric_pipeline = FunctionTransformer(skewed_imputer, validate=False)
            transformers.append(("numeric", numeric_pipeline, numeric_cols))

        # Low cardinality categorical transformer
        if low_card_cols:
            low_card_pipeline = Pipeline(
                [("imputer", SimpleImputer(strategy="most_frequent"))]
            )
            transformers.append(("low_card", low_card_pipeline, low_card_cols))

        # High cardinality categorical transformer
        if high_card_cols:
            high_card_pipeline = Pipeline(
                [("imputer", SimpleImputer(strategy="constant", fill_value="Unknown"))]
            )
            transformers.append(("high_card", high_card_pipeline, high_card_cols))

        # Create ColumnTransformer
        if not transformers:
            raise ValueError("No valid columns found for preprocessing")

        preprocessor = ColumnTransformer(
            transformers=transformers,
            remainder="drop",  # Drop any remaining columns
            sparse_threshold=0,  # Return dense array
        )

        # Store pipeline information
        self.pipeline_info = {
            "numeric": numeric_cols,
            "low_card": low_card_cols,
            "high_card": high_card_cols,
            "dropped": getattr(self, "sparse_columns", []),
            "all_processed": numeric_cols + low_card_cols + high_card_cols,
        }

        return preprocessor


class Preprocessor(
    SkewedImputerMixin,
    MissingValueAnalyzerMixin,
    ImputerPipelineMixin,
    BaseEstimator,
    TransformerMixin,
):
    def __init__(
        self,
        cardinality_threshold=10,
        skew_threshold=0.5,
        sparse_columns=[],
        retain_columns=[],
    ):
        self.cardinality_threshold = cardinality_threshold
        self.skew_threshold = skew_threshold
        self.sparse_columns = sparse_columns
        self.retain_columns = retain_columns

    def fit(self, X, y=None):
        """Fit the preprocessor to the training data"""
        if self.sparse_columns and self.sparse_columns in X.columns.to_list():
            X.drop(columns=self.sparse_columns, inplace=True, axis=1)
        if self.retain_columns:
            X = X[self.retain_columns]

        if not isinstance(X, pd.DataFrame):
            raise ValueError("Input must be a pandas DataFrame")

        print(f"🔄 Fitting DataPreprocessor on data with shape: {X.shape}")

        try:
            # Step 1: Analyze missing values and identify sparse columns
            print("Step 1: Analyzing missing values...")
            self.analyze(X)

            # Step 2: Determine numeric imputation strategies based on skewness
            print("Step 2: Determining numeric imputation strategies...")
            self._numeric_imputer_strategy(X, self.skew_threshold)

            # Step 3: Build and fit the preprocessing pipeline
            print("Step 3: Building preprocessing pipeline...")
            self.imputer_pipeline = self.build_imputer_pipeline(
                X, self.cardinality_threshold, self.skew_threshold
            )

            print("Step 4: Fitting pipeline...")
            self.imputer_pipeline.fit(X)

            print("✅ DataPreprocessor fitted successfully!")
            return self

        except Exception as e:
            print(f"❌ Error during fitting: {e}")
            raise

    def transform(self, X):
        """Transform the input data using the fitted pipeline"""
        if not hasattr(self, "imputer_pipeline"):
            raise ValueError("Pipeline not fitted. Call fit() first.")

        # Apply transformation
        X_transformed = self.imputer_pipeline.transform(X)

        # Get column names in the correct order
        all_cols = self.pipeline_info["all_processed"]

        # Validate dimensions
        if X_transformed.shape[1] != len(all_cols):
            print(f"Warning: Shape mismatch!")
            print(f"X_transformed shape: {X_transformed.shape}")
            print(f"Expected columns: {len(all_cols)}")
            print(f"Using generic column names...")
            # all_cols = [f"feature_{i}" for i in range(X_transformed.shape[1])]

        # Convert to DataFrame
        try:
            X_transformed_df = pd.DataFrame(
                X_transformed, columns=all_cols, index=X.index
            )
            X_transformed_df = X_transformed_df.astype(
                X.loc[:, X_transformed_df.columns].dtypes.to_dict()
            )
            print(f"✅ Successfully transformed to DataFrame: {X_transformed_df.shape}")
            return X_transformed_df

        except Exception as e:
            print(f"❌ Error creating DataFrame: {e}")
            return X_transformed

    def get_feature_names_out(self):
        """Get the names of the output features after transformation"""
        if hasattr(self, "pipeline_info"):
            return self.pipeline_info["all_processed"]
        else:
            raise ValueError("Pipeline not fitted yet. Call fit() first.")

    def fit_transform(self, X, y=None):
        """Fit the transformer and transform the data in one step"""
        return self.fit(X, y).transform(X)

    def get_pipeline_summary(self):
        """Get a summary of the preprocessing pipeline"""
        if not hasattr(self, "pipeline_info"):
            return "Pipeline not fitted yet."

        summary = {
            "Total features processed": len(self.pipeline_info["all_processed"]),
            "Numeric features": len(self.pipeline_info["numeric"]),
            "Low cardinality categorical": len(self.pipeline_info["low_card"]),
            "High cardinality categorical": len(self.pipeline_info["high_card"]),
            "Dropped features": len(self.pipeline_info["dropped"]),
        }

        return summary


In [ ]:
df_before_clean

In [ ]:
preprocessor = Preprocessor(skew_threshold=0.5, cardinality_threshold=10,sparse_columns=['Id'])
df_clean = preprocessor.fit_transform(df_before_clean)

# print(cleaned_df.dtypes)
df_clean.isnull().sum()

In [ ]:
df_clean

In [ ]:
df_clean.isnull().sum()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

yearbuiltbin = pd.cut(
    df_clean["YearBuilt"],
    bins=range(1870, 2015, 5),   # bins of 5 years
    labels=[f"{y}-{y+5}" for y in range(1870, 2010, 5)]
)
plt.figure(figsize=(12, 6))
sns.barplot(
    x=yearbuiltbin,
    y=df_clean['SalePrice'],
    errorbar=None,   # removes confidence intervals if not needed
    estimator="mean" # default is mean, but you can set "median"
)
plt.xticks(rotation=90)  # rotate labels if too many years
plt.title("Average Sale Price by Year Built", fontsize=14)
plt.xlabel("Year Built")
plt.ylabel("Average Sale Price")
plt.show()


In [ ]:
summary = df_clean.isnull().sum()
summary[summary > 0]

## **Feature analysis with Data Visualization**

### **Univariate Analysis**

#### Numerical Feature

In [ ]:
num_columns = df_clean.select_dtypes(include=['int64','float64','int32']).columns.to_list()
n = len(num_columns)
n_cols = int(np.ceil(np.sqrt(n)))
n_rows = int(np.ceil(n / n_cols))


# Flatten for indexing
# axes = axes.flatten()

n_cols = int(np.ceil(np.sqrt(n)))
n_rows = int(np.ceil(n / n_cols))
fig = plt.figure(figsize=(4*n_cols, 3*n_rows*2))
gs = fig.add_gridspec(n_rows*2, n_cols, hspace=0.4)

for i, col in enumerate(num_columns):
    r = (i // n_cols) * 2  # row start (two rows per feature)
    c = i % n_cols

    # Determine shared x-axis limits
    xmin, xmax = df_clean[col].min(), df_clean[col].max()

    # Boxplot (top)
    ax_box = fig.add_subplot(gs[r, c])
    sns.boxplot(x=df_clean[col], ax=ax_box, color="lightblue")
    ax_box.set_title(f"{col} - Box")
    ax_box.set_xlim(xmin, xmax)
    ax_box.set_xlabel("")

    # Histogram (bottom)
    ax_hist = fig.add_subplot(gs[r+1, c], sharex=ax_box)
    sns.histplot(df_clean[col], kde=True, ax=ax_hist, color="steelblue")
    ax_hist.set_title(f"{col} - Hist")
    ax_hist.set_xlim(xmin, xmax)

plt.tight_layout()
plt.show()

**The plots show that none of the features, including the target `SalePrice`, follow a normal distribution. These need transformation into near-normal distributions using methods such as Log, Box-Cox, or Yeo–Johnson. Before applying transformations, identify the most correlated features**

#### Categorical Feature

In [ ]:
category_columns = df_clean.select_dtypes(include=['object']).columns.to_list()
num_columns = df_clean.select_dtypes(include=['int64','float64','int32']).columns.to_list()
high_cardinal_cat_columns = [col for col in category_columns if df_clean[col].nunique() > 10]
low_cardinal_cat_columns = [col for col in category_columns if df_clean[col].nunique() <= 10]

In [ ]:
category_columns[0]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

n = len(category_columns)

# Determine rows and columns dynamically
n_cols = 4
n_rows = int(np.ceil(n / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten()

for i, col in enumerate(category_columns):
    sns.countplot(x=col, data=df_clean, ax=axes[i], palette='pastel')
    axes[i].set_title(f'{col} - Count')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)

# Remove unused axes
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
fig.savefig("new.png")
plt.show()

**Features that are more evenly distributed have a higher potential to serve as effective predictors.**

The countplots visualize the distribution of values within each categorical feature. Key observations include:

*   **Dominant Categories:** Many features have a single category that appears much more frequently than others (e.g., 'MSZoning' is dominated by 'RL'). These features may have limited predictive power on their own but could be useful when combined with other features.
*   **Diverse Distributions:** Some features, like 'Neighborhood', 'HouseStyle', and exterior related columns, show a more even distribution across their categories. These features are likely to be more informative for predicting `SalePrice`.
*   **Potential Issues:** Some categories have very few instances. These rare categories might need to be grouped with other categories or handled specifically during feature engineering to avoid issues with model training.

Overall, these plots help us understand the composition of our categorical data and identify potential areas for feature engineering or further analysis.

### Target Analysis

In [ ]:
sbn.histplot(df_clean['SalePrice'],kde=True)

**Insights**
- `SalePrice` data points is right skewed
- This violates one of the key assumptions of regression, which may affect the validity of the model.
- Hence, transformations such as Box-Cox or Log Transformation will be applied in the forthcoming stage.

### **Correlation Analysis**

#### Numerical Features

In [ ]:
num_features = df_clean.select_dtypes(include=['int64','float64','int32']).columns.to_list()

In [ ]:
correlation = df_clean[num_features].corr(method='spearman')['SalePrice'].sort_values(ascending=False)
corr_num_df = correlation.to_frame(name="relevance_with_price")
plt.figure(figsize=(12,8))
sns.heatmap(corr_num_df, annot=True, cmap='coolwarm')
plt.title("Spearman Correlation with SalePrice")
plt.show()

In [ ]:
relevant_num_features = corr_num_df[(corr_num_df['relevance_with_price'] > 0.7) & (corr_num_df.index != "SalePrice")].index.to_list()
relevant_num_features

#### Categorical Features

| Method                 | Feature Type       | Target Type                     | Pros                              | Cons                            |
| ---------------------- | ------------------ | ------------------------------- | --------------------------------- | ------------------------------- |
| ANOVA / F-test         | Categorical        | Numeric                         | Simple, interpretable             | Only significance, not strength |
| Correlation ratio η²   | Categorical        | Numeric                         | Measures explained variance       | Slightly complex                |
| Mutual Information     | Categorical        | Numeric                         | Captures non-linear relationships | Needs encoding, no direction    |
| Point Biserial         | Binary categorical | Numeric                         | Simple correlation                | Only binary categories          |
| Spearman / Kendall     | Ordinal            | Numeric                         | Captures monotonic trends         | Not for nominal                 |
| Tree-based importance  | Any                | Numeric                         | Non-linear, mixed types           | Model-dependent                 |
| Permutation importance | Any                | Numeric                         | Model-agnostic                    | Slower                          |
| Chi-squared            | Categorical        | Categorical (or binned numeric) | Statistical dependence            | Needs target binning            |


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Example categorical features
# Determine number of rows and columns for subplots
n_features = len(category_columns)
n_cols = 2  # number of columns in the grid
n_rows = (n_features + n_cols - 1) // n_cols  # ceil division

fig, axes = plt.subplots(n_rows, 3, figsize=(12, n_rows*4))
axes = axes.flatten()  # flatten in case of single row/column

for i, cat in enumerate(category_columns):
    sns.boxplot(x=df_clean[cat], y=df_clean['SalePrice'], ax=axes[i])
    axes[i].set_title(f'{cat} vs SalePrice')
    axes[i].tick_params(axis='x', rotation=45)

# Turn off any unused subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
sbn.boxplot(x=df_clean['BldgType'],y=df_clean['SalePrice'])
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
sbn.boxplot(x=df_clean['Neighborhood'],y=df_clean['SalePrice'])
plt.xticks(rotation=45)
plt.show()

| Method | Type of Categorical Variable | Type of Numeric Variable | When to Use |
|---------------------------|-----------------------------|-------------------------|--------------------------------------|
| Point Biserial Correlation | Binary | Continuous | Two categories |
| ANOVA | Nominal | Continuous | More than two categories |
| Kruskal-Wallis H Test | Nominal | Continuous | Non-parametric, more than two categories |
| Chi-Squared Test | Nominal | Categorical (binned) | Relationship between two categorical variables |
| Spearman's Rank Correlation| Ordinal | Continuous | Ordered categories |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import f_oneway


class FeatureSelector(BaseEstimator, TransformerMixin):
    """
    Combines numeric and categorical correlation analysis for regression.

    Parameters
    ----------
    target : str
        Name of the target column.
    eta_threshold : float, default=0.25
        Minimum correlation ratio (eta squared) to select categorical features.
    pvalue_threshold : float, default=0.05
        ANOVA p-value threshold to select categorical features.
    visualize : bool, default=False
        If True, plots numeric and categorical correlations.
    """

    def __init__(
        self,
        target=None,
        num_feat_corr_threshold=0.7,
        eta_threshold=0.25,
        pvalue_threshold=0.05,
        visualize=False,
        chosen_features=[]
    ):
        self.target = target
        self.eta_threshold = eta_threshold
        self.num_feat_corr_threshold = num_feat_corr_threshold
        self.pvalue_threshold = pvalue_threshold
        self.visualize = visualize
        self.selected_features_ = []
        self.chosen_features = chosen_features
        self.num_corr_df_ = None
        self.cat_corr_df_ = None

    @staticmethod
    def correlation_ratio(categories, measurements):
        fcat, _ = pd.factorize(categories)
        mean_total = measurements.mean()
        ss_between = sum(
            len(measurements[fcat == i])
            * (measurements[fcat == i].mean() - mean_total) ** 2
            for i in np.unique(fcat)
        )
        ss_total = sum((measurements - mean_total) ** 2)
        return ss_between / ss_total if ss_total != 0 else 0

    def fit(self, X, y=None):
        # Numeric correlation
        if self.chosen_features:
            return self
        self.num_corr_df_ = (
            X.corr(method="spearman", numeric_only=True)[self.target]
            .sort_values(ascending=False)
            .to_frame(name=f"Correlation with {self.target}")
        )

        # Extract correlation values as Series for feature selection
        corr_series = X.corr(method="spearman", numeric_only=True)[self.target]
        self.selected_features_ = corr_series[
            (corr_series.abs() > self.num_feat_corr_threshold)
            & (corr_series.index != self.target)
        ].index.tolist()

        # Categorical correlation
        results = []
        cat_cols = X.select_dtypes(include="object").columns.tolist()
        for col in cat_cols:
            groups = [X[self.target][X[col] == lvl] for lvl in X[col].dropna().unique()]
            if len(groups) < 2:
                continue
            f_stat, p_val = f_oneway(*groups)
            eta = self.correlation_ratio(X[col], X[self.target])
            results.append(
                {
                    "Feature": col,
                    "F_stat": f_stat,
                    "p_value": p_val,
                    "Correlation_Ratio": eta,
                }
            )

        df_results = pd.DataFrame(results)
        self.cat_corr_df_ = df_results.sort_values("Correlation_Ratio", ascending=False)

        # Feature selection
        if not df_results.empty:
            mask = (df_results["Correlation_Ratio"] > self.eta_threshold) & (
                df_results["p_value"] < self.pvalue_threshold
            )
            filtered = df_results[mask]
            self.selected_features_.extend(filtered["Feature"].tolist())

        if self.visualize:
            self.plot()

        return self

    def transform(self, X):
        if not self.selected_features_ and not self.chosen_features:
            return X
        if self.chosen_features:
          return X[self.chosen_features]
        else:
          cols_to_keep = self.selected_features_

        return X[cols_to_keep+[self.target]]

    def plot(self):

        # Numeric
        plt.figure(figsize=(15, 8))
        plt.subplot(121)
        sns.heatmap(self.num_corr_df_, annot=True, cmap="coolwarm")
        plt.title(f"Numeric Correlation with {self.target}")

        # Categorical
        plt.subplot(122)
        if not self.cat_corr_df_.empty:
            sns.barplot(
                x="Correlation_Ratio",
                y="Feature",
                data=self.cat_corr_df_,
                palette="viridis",
            )
            plt.title(f"Categorical Correlation with {self.target}")
            plt.xlabel("Correlation Ratio (η²)")
            plt.ylabel("Categorical Feature")
        plt.tight_layout()
        plt.show()


In [ ]:
feature_selector = FeatureSelector(visualize=True,target="SalePrice")
feature_selector.fit_transform(df_clean)

In [ ]:
df_feat_extracted = pd.concat([df_clean[feature_selector.selected_features_],df_clean["SalePrice"]],axis=1)

In [ ]:
df_feat_extracted.isnull().sum()

In [ ]:

corr = df_feat_extracted.corr(numeric_only=True)['SalePrice'].sort_values()

# Plot as bar chart
ax = corr.plot(kind='bar', figsize=(12, 8))

# Annotate bars
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", label_type="edge", fontsize=10)

plt.title("Correlation of Numerical Features with SalePrice", fontsize=14)
plt.ylabel("Correlation Coefficient")
plt.xticks(rotation=45, ha='right')
plt.show()

### Feature Explanation

### 🏘️ Neighborhood

* **Type**: Categorical
* **Description**: Physical locations within Ames city limits.
* **Examples**:

  * `Blmngtn` → Bloomington Heights
  * `Gilbert` → Gilbert
  * `StoneBr` → Stone Brook

---

### 🪟 ExterQual

* **Type**: Ordinal (categorical with order)
* **Description**: Evaluates the quality of the material on the exterior.
* **Values**:

  * `Ex` → Excellent
  * `Gd` → Good
  * `TA` → Typical/Average
  * `Fa` → Fair
  * `Po` → Poor

---

### 🏠 BsmtQual

* **Type**: Ordinal
* **Description**: Evaluates the height of the basement.
* **Values**:

  * `Ex` → Excellent (100+ inches)
  * `Gd` → Good (90–99 inches)
  * `TA` → Typical (80–89 inches)
  * `Fa` → Fair (70–79 inches)
  * `Po` → Poor (<70 inches)
  * `NA` → No Basement

---

### 👩‍🍳 KitchenQual

* **Type**: Ordinal
* **Description**: Kitchen quality rating.
* **Values**:

  * `Ex` → Excellent
  * `Gd` → Good
  * `TA` → Typical/Average
  * `Fa` → Fair
  * `Po` → Poor

---

### 🚗 GarageFinish

* **Type**: Categorical
* **Description**: Interior finish of the garage.
* **Values**:

  * `Fin` → Finished
  * `RFn` → Rough Finished
  * `Unf` → Unfinished
  * `NA` → No Garage

---

### 🧱 Foundation

* **Type**: Categorical
* **Description**: Type of foundation of the house.
* **Values**:

  * `BrkTil` → Brick & Tile
  * `CBlock` → Cinder Block
  * `PConc` → Poured Concrete
  * `Slab` → Slab
  * `Stone` → Stone
  * `Wood` → Wood

---

### 📊 OverallQual

* **Type**: Ordinal (1–10 scale)
* **Description**: Rates the overall material and finish of the house.
* **Scale**:

  * `10` → Very Excellent
  * `9` → Excellent
  * `8` → Very Good
  * `7` → Good
  * `6` → Above Average
  * `5` → Average
  * `4` → Below Average
  * `3` → Fair
  * `2` → Poor
  * `1` → Very Poor

---

### 📏 GrLivArea

* **Type**: Numerical (continuous)
* **Description**: Above grade (ground) living area square feet.
* **Example**: `1500` sq. ft., `2500` sq. ft.

---

### 💰 SalePrice

* **Type**: Numerical (continuous, target variable)
* **Description**: Sale price of the house in USD.
* **Example**: `200000`, `340000`

---


### **Bivariate Analysis**

In [ ]:
plt.figure(figsize=(12, 8))
# sns.scatterplot(x='OverallQual', y='SalePrice', data=df_clean)
plt.subplot(1, 3, 1)
sns.regplot(x='OverallQual', y='SalePrice', data=df_feat_extracted)
plt.subplot(1, 3, 2)
sns.scatterplot(x='OverallQual', y='SalePrice', data=df_feat_extracted)
# plt.show()
plt.subplot(1, 3, 3)
sns.scatterplot(x='GrLivArea', y='SalePrice', data=df_feat_extracted)
plt.show()

**Above plot depicts strong relationshsip between OverQual and SalePrice**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

categorical_cols = ["Neighborhood","ExterQual","BsmtQual","KitchenQual","GarageFinish","Foundation"]

# Define subplot grid
n_cols = 3
n_rows = (len(categorical_cols) +1) // n_cols
fig, axes = plt.subplots(n_rows+1, n_cols, figsize=(15, 10))
axes = axes.flatten()  # Flatten for easy indexing

for i, col in enumerate(categorical_cols):
    sns.violinplot(x=col, y="SalePrice", data=df_feat_extracted, ax=axes[i])
    axes[i].set_title(f"{col} vs SalePrice")
    axes[i].tick_params(axis='x', rotation=45)

# Remove any empty subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


### Multivariate Analysis

In [ ]:
sns.lmplot(
    data=df_feat_extracted,
    x="GrLivArea", y="SalePrice",
    hue="OverallQual", scatter_kws={"alpha":0.5}
)
plt.show()


- There is a general positive correlation between living area and sales price, meaning larger homes tend to have higher sales prices.
- Homes with higher OverallQual ratings (e.g., 8, 9, 10) are mostly clustered at higher sales prices and larger living areas, indicating better quality homes are more expensive.
- Lower-rated homes (e.g., 1, 2, 3) are concentrated at lower sales prices and smaller living areas.
- A cyan triangular confidence interval suggests a trend line, with most data points falling within or near this area, reinforcing the correlation.
- Outliers exist, such as a few homes with large living areas but relatively low sales prices, possibly indicating other influencing factors like location or condition.

In [ ]:
g = sns.FacetGrid(df_feat_extracted, col="OverallQual", col_wrap=5, height=3)
g.map(sns.scatterplot, "ExterQual", "SalePrice")
plt.show()


In [ ]:
g = sns.FacetGrid(df_feat_extracted, col="OverallQual", col_wrap=5, height=3)
g.map(sns.boxplot, "ExterQual", "SalePrice")
plt.show()


- **OverallQual Impact**: Sales prices generally increase with higher OverallQual ratings. Homes with OverallQual 1-3 have very low median prices \(below \$100,000\)  while those with OverallQual $8-10$ show median prices ranging from \$200,000 to over \$400,000.

- **ExterQual Influence**: Within each OverallQual category, exterior quality (Fa to Ex) shows a trend where better exterior quality (Gd, Ex) tends to correlate with higher sales prices, though the effect is more pronounced in higher OverallQual groups (e.g., 8-10).
- **Variability**: Lower OverallQual ratings (1-5) show less variability and lower price ranges, while higher ratings (6-10) exhibit greater spread, with some homes reaching up to $600,000-$700,000, especially with excellent exterior quality.
- **Outliers**: Notable outliers exist, particularly in OverallQual 5-10, where some homes exceed $500,000, suggesting exceptional cases influenced by factors beyond quality ratings.

This indicates that both overall quality and exterior quality significantly affect sales prices, with stronger effects at higher quality levels.

**ExterQual is associated with OverallQual rather than causing it. To decide whether both should be kept in the regression model, we need to check multicollinearity using VIF**

In [ ]:
df_feat_extracted['Foundation'].value_counts()

In [ ]:
g = sbn.FacetGrid(df_feat_extracted, col="KitchenQual", col_wrap=4, height=4)
g.map(sbn.regplot, "OverallQual", "SalePrice")
plt.show()

In [ ]:
sbn.scatterplot(data=df_feat_extracted, x="OverallQual", y="SalePrice", hue="Foundation")
plt.show()

In [ ]:
g = sbn.FacetGrid(df_feat_extracted, col="OverallQual", col_wrap=4, height=4)
g.map(sns.regplot, "GrLivArea", "SalePrice", scatter_kws={'s':10}, line_kws={'color':'red'})
plt.show()

The **light red space around the regression line** is the **confidence interval (CI)** of the regression estimate.

* By default, `sns.regplot` shows a **95% confidence interval**.
* It represents the range in which the true regression line is likely to fall.
* Narrow bands → more certainty in the regression estimate.
* Wider bands → more variability or fewer data points.


In [ ]:
sns.pairplot(df_feat_extracted[['OverallQual', 'GrLivArea', 'SalePrice']])
plt.show()

### Feature Encoding

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline, make_pipeline
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin


# -----------------------------
# 1) Feature Groups
# -----------------------------
# ordinal_features = ["ExterQual", "BsmtQual", "KitchenQual"]
# categorical_features = ["Neighborhood", "GarageFinish", "Foundation"]
# numerical_features = ["OverallQual", "GrLivArea"]

# ordinal_mappings = {
#     "ExterQual": {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
#     "BsmtQual": {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
#     "KitchenQual": {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
# }


# -----------------------------
# 5) Decode Helper (works with both Pipeline and make_pipeline)
# -----------------------------
def decode(pipeline, encoder_step, feature, encoded_value):
    """
    Recover original category from encoded integer.
    Works with both Pipeline and make_pipeline.

    For make_pipeline, step names are auto-generated like:
    - 'ordinalmapper' (lowercase, no underscores)
    - 'categoricallabelencoder'
    """
    # Handle both named steps (Pipeline) and auto-generated names (make_pipeline)
    if hasattr(pipeline, "named_steps"):
        # For Pipeline with named steps
        if encoder_step in pipeline.named_steps:
            encoders = pipeline.named_steps[encoder_step].encoders
        else:
            # For make_pipeline, try auto-generated names
            step_names = list(pipeline.named_steps.keys())
            print(f"Available steps: {step_names}")
            raise ValueError(
                f"Step '{encoder_step}' not found. Available steps: {step_names}"
            )
    else:
        # For very old sklearn versions or custom pipelines
        encoders = pipeline.steps[encoder_step][1].encoders

    if feature not in encoders:
        raise ValueError(f"Feature '{feature}' not found in encoders.")

    le = encoders[feature]
    if encoded_value < 0 or encoded_value >= len(le.classes_):
        raise ValueError(f"Encoded value {encoded_value} out-of-range for '{feature}'.")

    return le.inverse_transform([encoded_value])[0]


def decode_make_pipeline(pipeline, feature, encoded_value):
    """
    Simplified decode function specifically for make_pipeline.
    Automatically finds the CategoricalLabelEncoder step.
    """
    # Find the CategoricalLabelEncoder step
    encoder_step = None
    for step_name, step_obj in pipeline.named_steps.items():
        if isinstance(step_obj, CategoricalLabelEncoder):
            encoder_step = step_obj
            break

    if encoder_step is None:
        raise ValueError("CategoricalLabelEncoder not found in pipeline")

    if feature not in encoder_step.encoders:
        raise ValueError(f"Feature '{feature}' not found in encoders.")

    le = encoder_step.encoders[feature]
    if encoded_value < 0 or encoded_value >= len(le.classes_):
        raise ValueError(f"Encoded value {encoded_value} out-of-range for '{feature}'.")

    return le.inverse_transform([encoded_value])[0]


class OrdinalEncoderDF(BaseEstimator, TransformerMixin):
    def __init__(self, cols, categories):
        self.cols = cols
        self.categories = categories
        self.encoder = OrdinalEncoder(categories=self.categories)

    def fit(self, X, y=None):
        self.encoder.fit(X[self.cols])
        return self

    def transform(self, X):
        X_new = X.copy()
        print(self.encoder.transform(X[self.cols]))
        X_new[self.cols] = self.encoder.transform(X[self.cols])
        X_new[self.cols] = X_new[self.cols].astype(int)
        return X_new


class FrequencyEncoderDF(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols
        self.freq_maps_ = {}

    def fit(self, X, y=None):
        self.cols_ = self.cols or X.select_dtypes(include="object").columns.tolist()
        for col in self.cols_:
            self.freq_maps_[col] = X[col].value_counts(normalize=True)
        return self

    def transform(self, X):
        X_new = X.copy()
        for col in self.cols_:
            X_new[col + "_freq"] = X_new[col].map(self.freq_maps_[col]).fillna(0)
        return X_new


# -------------------------
# 2. OneHot Encoder Wrapper
# -------------------------
class OneHotEncoderDF(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.encoder = OneHotEncoder(drop="first",handle_unknown="ignore", sparse_output=False)
        self.feature_names_ = []

    def fit(self, X, y=None):
        self.encoder.fit(X[self.cols])
        self.feature_names_ = self.encoder.get_feature_names_out(self.cols)
        return self

    def transform(self, X):
        X_new = X.copy()

        # Transform and get dense array
        data = self.encoder.transform(X_new[self.cols])

        # Debug information
        print(f"++++++++++++++++++++++++++++++++  {self.__class__.__name__} =================================")
        print(f"Transformed data shape: {data.shape}")
        print(f"Feature names: {self.feature_names_}")
        print(f"Number of feature names: {len(self.feature_names_)}")

        # Create DataFrame with encoded features
        encoded = pd.DataFrame(
            data,
            columns=self.feature_names_,
            index=X_new.index,
        )

        # Remove original columns and add encoded ones
        X_new = X_new.drop(columns=self.cols)
        X_new = pd.concat([X_new, encoded.apply(round).astype("int")], axis=1)

        return X_new


# -------------------------
# 3. Target Encoder Wrapper
# -------------------------
class DataFrameTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Target Encoder for regression. Compatible with sklearn pipelines.
    Accepts X (DataFrame) and y (Series) in fit, returns DataFrame in transform.
    """

    def __init__(self, cols=None, target=None, smoothing=10):
        self.cols = cols
        self.smoothing = smoothing
        self.target = target
        self.mapping_ = {}
        self.global_mean_ = None

    def fit(self, X):
        self.cols_ = self.cols or X.select_dtypes(include="object").columns.tolist()
        if not isinstance(X, pd.DataFrame):
            raise ValueError("X must be a pandas DataFrame.")
        if self.target is None:
            raise ValueError("y must be provided.")
        # if self.use_mapping:

        if self.target in X.columns:
            y = X[self.target]
        else:
            return self

        self.global_mean_ = y.mean()

        for col in self.cols_:
            stats = X.groupby(col).size().to_frame("count")
            means = X.groupby(col).apply(lambda df: y[df.index].mean())
            stats["mean"] = means
            smoothing = 1 / (1 + np.exp(-(stats["count"] - 1) / self.smoothing))
            self.mapping_[col] = (
                self.global_mean_ * (1 - smoothing) + stats["mean"] * smoothing
            ).apply(round)

        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("X must be a pandas DataFrame.")

        X_new = X.copy()
        for col in self.cols_:
            X_new[col] = (
                X_new[col].map(self.mapping_[col]).fillna(self.global_mean_)
            )

        return X_new


def create_encoder_pipeline(
    ordinal_categories,
    ordinal_features,
    low_cardinal_nominal_features,
    high_cardinal_nominal_features,
    target,
):

    pipe = Pipeline(
        [
            ("ordinal", OrdinalEncoderDF(ordinal_features, ordinal_categories)),
            ("low_card_ohe", OneHotEncoderDF(low_cardinal_nominal_features)),
            (
                "high_card_target",
                DataFrameTargetEncoder(high_cardinal_nominal_features, target=target),
            ),
        ]
    )

    return pipe


In [ ]:
ordinal_features = ["ExterQual", "BsmtQual", "KitchenQual","GarageFinish"]
ordinal_categories = [
    ["Po", "Fa", "TA", "Gd", "Ex"],        # ExterQual
    ["NA", "Po", "Fa", "TA", "Gd", "Ex"],  # BsmtQual
    ["Po", "Fa", "TA", "Gd", "Ex"],        # KitchenQual
    [ "NA", "Unf", "RFn", "Fin"]           # GarageFinish
]
high_cardinal_nominal_features = [ 'Neighborhood']
low_cardinal_nominal_features = ["Foundation"]

encoder = create_encoder_pipeline(ordinal_categories,ordinal_features,low_cardinal_nominal_features,high_cardinal_nominal_features,target="SalePrice")

In [ ]:
df_encoded = encoder.fit_transform(df_feat_extracted)

### Mulicollinearity Analysis of Features

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = add_constant(df_encoded.drop("SalePrice",axis=1))  # add intercept
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                   for i in range(X.shape[1])]
print(vif_data)


**Based on the results, there is no evidence of multicollinearity among the features**

| VIF Value | Interpretation           | Action                     |
| --------- | ------------------------ | -------------------------- |
| **1**     | No correlation           | ✅ Safe                     |
| **1 – 3** | Low correlation          | ✅ Acceptable               |
| **3 – 5** | Moderate correlation     | ⚠️ Watch closely           |
| **> 5**   | High multicollinearity   | ❌ Consider fixing          |
| **> 10**  | Severe multicollinearity | ❌ Definitely remove/adjust |


### Feature Selection

**Based on the above results , we need to keep all the above :features as final predictors**

### **Feature Transformation**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import PowerTransformer


class FeatureTransformer(BaseEstimator, TransformerMixin):
    """
    Continuous Feature Transformer for regression or general numeric datasets.

    Automatically detects continuous numerical features (by minimum unique values),
    calculates skewness, and applies appropriate transformations to reduce skew:

    - log1p: for high positive skew and strictly positive values
    - Box-Cox: for moderate positive skew and strictly positive values
    - Yeo-Johnson: for negative skew or features containing zeros/negative values
    - None: for features with low skewness

    Can visualize the original vs transformed distributions using seaborn.

    Parameters
    ----------
    skew_threshold : float, default=0.5
        Minimum absolute skewness required to apply any transformation.
    # log_skew_threshold : float, default=1.0
        # Skew above this threshold uses log1p transformation (if all values > 0).
    min_unique : int, default=30
        Minimum distinct values to consider a feature continuous.
    visualize : bool, default=True
        If True, will visualize original vs transformed distributions after fit_transform.

    Attributes
    ----------
    continuous_features_ : list
        List of features identified as continuous based on min_unique.
    transformers_ : dict
        Mapping of feature name -> (transformation method, fitted transformer object)
    """

    def __init__(self, skew_threshold=0.5, min_unique=30, visualize=True):
        self.skew_threshold = skew_threshold
        self.min_unique = min_unique
        self.visualize = visualize
        self.continuous_features_ = []
        self.transformers_ = {}

    def fit(self, X, y=None):
        numeric_features = X.select_dtypes(include=np.number).columns.tolist()
        self.continuous_features_ = [
            col
            for col in numeric_features
            if X[col].nunique() > self.min_unique and not col.endswith("_encoded")
        ]
        for col in self.continuous_features_:
            skewness = X[col].skew()
            col_min = X[col].min()

            if abs(skewness) > self.skew_threshold:
                if skewness > 0.5 and col_min > 0:
                    self.transformers_[col] = ("log1p", None)
                elif 0 < skewness <= 0.5 and col_min > 0:
                    pt = PowerTransformer(method="box-cox").fit(X[[col]])
                    self.transformers_[col] = ("box-cox", pt)
                else:
                    pt = PowerTransformer(method="yeo-johnson").fit(X[[col]])
                    self.transformers_[col] = ("yeo-johnson", pt)
            else:
                self.transformers_[col] = ("none", None)
        return self

    def transform(self, X, y=None):
        X_new = X.copy()
        for col, (method, transformer) in self.transformers_.items():
            if method == "log1p":
                X_new[col] = np.log1p(X_new[col])
            elif method in ["box-cox", "yeo-johnson"]:
                X_new[col] = transformer.transform(X_new[[col]])
        if self.visualize:
            self.visualize_distributions(X, X_new)

        return X_new

    def inverse_transform(self, X):
        X_orig = X.copy()
        for col, (method, transformer) in self.transformers_.items():
            if method == "log1p":
                X_orig[col] = np.expm1(X_orig[col])
            elif method in ["box-cox", "yeo-johnson"]:
                X_orig[col] = transformer.inverse_transform(X_orig[[col]])
        return X_orig

    def visualize_distributions(self, X, X_transformed):
        """Visualize all continuous features before and after transformation using seaborn."""
        n_features = len(self.continuous_features_)
        if n_features == 0:
            print("No continuous features detected for visualization.")
            return

        n_cols = 2  # before and after
        n_rows = n_features
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))

        if n_rows == 1:
            axes = np.expand_dims(axes, axis=0)  # make it 2D for consistency

        for i, col in enumerate(self.continuous_features_):
            sns.histplot(X[col], bins=30, kde=True, color="skyblue", ax=axes[i, 0])
            axes[i, 0].set_title(f"Original: {col}\nSkew={X[col].skew():.2f}")

            sns.histplot(
                X_transformed[col], bins=30, kde=True, color="lightgreen", ax=axes[i, 1]
            )
            axes[i, 1].set_title(
                f"Transformed: {col}\nSkew={X_transformed[col].skew():.2f}"
            )

        plt.tight_layout()
        plt.show()


In [ ]:
feature_transformer = FeatureTransformer()
df_transformed = feature_transformer.fit_transform(df_encoded)
df_transformed

#### Target

### Feature Scaling

In [ ]:
sbn.histplot(df_transformed[target],kde=True)
plt.show()

**However the fact the `SalePrice` is not scaled, Standardizing it is not necessary**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler


class FeatureScaler(BaseEstimator, TransformerMixin):
    """
    Scale continuous numeric features in a DataFrame using different scaling strategies.

    This transformer automatically detects continuous numeric columns
    and applies the chosen scaling method. It supports inverse_transform
    for integration in pipelines. It can also visualize before/after scaling.

    Parameters
    ----------
    method : str, default="auto"
        Scaling strategy to apply:
        - "auto": automatically chooses scaler based on skew/outliers.
        - "standard": use StandardScaler (mean=0, std=1).
        - "minmax": use MinMaxScaler (scale to [0,1]).
        - "robust": use RobustScaler (median=0, IQR=1, robust to outliers).

    skew_threshold : float, default=0.5
        Absolute skew above which 'auto' mode may use RobustScaler.

    min_unique : int, default=30
        Minimum number of unique values for a column to be considered continuous.

    visualize_flag : bool, default=True
        Whether to plot boxplots before and after scaling in fit_transform.

    Attributes
    ----------
    continuous_features_ : list
        List of continuous numeric columns detected in the DataFrame.

    scalers_ : dict
        Dictionary mapping each continuous column to its fitted scaler.

    Notes on when to use each scaler
    --------------------------------
    - StandardScaler: Use when features are roughly Gaussian and free of extreme outliers.
    - MinMaxScaler: Use when you want all features in the [0,1] range, e.g., for neural networks.
    - RobustScaler: Use when features contain outliers or are highly skewed.
    - Auto mode: Automatically applies RobustScaler for skewed or repeated-value features,
      otherwise StandardScaler.
    """

    def __init__(
        self,
        method="auto",
        target=None,
        skew_threshold=0.5,
        min_unique=30,
        visualize=True,
    ):
        self.target = target
        self.method = method
        self.skew_threshold = skew_threshold
        self.min_unique = min_unique
        self.visualize = visualize
        self.continuous_features_ = []
        self.scalers_ = {}

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("Input must be a pandas DataFrame.")

        numeric_features = X.select_dtypes(include=np.number).columns.tolist()
        self.continuous_features_ = [
            col for col in numeric_features if X[col].nunique() > self.min_unique
        ]
        if self.target and self.target in self.continuous_features_:
            self.continuous_features_.remove(self.target)
        for col in self.continuous_features_:
            skewness = X[col].skew()

            if self.method == "standard":
                scaler = StandardScaler().fit(X[[col]])
            elif self.method == "minmax":
                scaler = MinMaxScaler().fit(X[[col]])
            elif self.method == "robust":
                scaler = RobustScaler().fit(X[[col]])
            else:  # auto
                if (
                    abs(skewness) > self.skew_threshold
                    or X[col].nunique() / X.shape[0] < 0.05
                ):
                    scaler = RobustScaler().fit(X[[col]])
                else:
                    scaler = StandardScaler().fit(X[[col]])

            self.scalers_[col] = scaler
        return self

    def transform(self, X):
        X_new = X.copy()
        for col, scaler in self.scalers_.items():
            X_new[col] = scaler.transform(X_new[[col]])
        if self.visualize:
            self.visualize_distribution(X, X_new)
        return X_new

    def inverse_transform(self, X):
        X_orig = X.copy()
        for col, scaler in self.scalers_.items():
            X_orig[col] = scaler.inverse_transform(X_orig[[col]])
        return X_orig

    def visualize_distribution(self, X, X_scaled=None):
        """Visualize boxplots before and after scaling for continuous features."""
        if len(self.continuous_features_) == 0:
            print("No continuous features detected.")
            return

        X_scaled = X_scaled if X_scaled is not None else self.transform(X)
        n_features = len(self.continuous_features_)

        fig, axes = plt.subplots(n_features, 2, figsize=(12, 4 * n_features))
        if n_features == 1:
            axes = np.expand_dims(axes, axis=0)  # Make 2D array for consistent indexing

        for i, col in enumerate(self.continuous_features_):
            sns.boxplot(x=X[col], ax=axes[i, 0], color="skyblue")
            axes[i, 0].set_title(f"Original: {col}")

            sns.boxplot(x=X_scaled[col], ax=axes[i, 1], color="lightgreen")
            axes[i, 1].set_title(f"Scaled: {col}")

        plt.tight_layout()
        plt.show()


In [ ]:
feature_scaler = FeatureScaler(target="SalePrice")
df_scaled = feature_scaler.fit_transform(df_transformed)
df_scaled

### Processor Pipeline for Trained Data

In [ ]:

ordinal_features = ["ExterQual", "BsmtQual", "KitchenQual","GarageFinish"]
ordinal_categories = [
    ["Po", "Fa", "TA", "Gd", "Ex"],        # ExterQual
    ["NA", "Po", "Fa", "TA", "Gd", "Ex"],  # BsmtQual
    ["Po", "Fa", "TA", "Gd", "Ex"],        # KitchenQual
    [ "NA", "Unf", "RFn", "Fin"]           # GarageFinish
]
high_cardinal_nominal_features = [ 'Neighborhood']
low_cardinal_nominal_features = ["Foundation"]

encoder = create_encoder_pipeline(ordinal_categories,ordinal_features,low_cardinal_nominal_features,high_cardinal_nominal_features,target="SalePrice")
target = "SalePrice"

def create_full_pipeline(target):
    full_pipeline = Pipeline([
        ("preprocessor",Preprocessor(cardinality_threshold=10, skew_threshold=0.5)),
        ("selector",FeatureSelector(target=target,visualize=False)),
        ("encoder",encoder),
        ("transformer", FeatureTransformer(visualize=False)),
        ("scaler", FeatureScaler(target=target))

    ])

    return full_pipeline

In [ ]:
train_pipeline = create_full_pipeline(target)
train_pipeline.fit(train_df)
df_train = train_pipeline.transform(train_df)
df_train

In [ ]:
train_pipeline.named_steps['encoder'].named_steps["high_card_target"].mapping_

### Full Pipeline for Test Data

In [ ]:
class SelectColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns or []

    def fit(self, X, y=None):
        return self  # nothing to learn

    def transform(self, X):
        return X[self.columns]


In [ ]:
selected_features = train_pipeline.named_steps['selector'].selected_features_
selected_features

In [ ]:
test_pipeline = Pipeline([
    ("preprocessor",Preprocessor(cardinality_threshold=10, skew_threshold=0.5)),
    ("drop_columns", SelectColumns(columns=selected_features)),
    ("encoder",encoder),
    ("transformer", FeatureTransformer()),
    ("scaler", FeatureScaler(target="SalePrice"))

])

### Test Data Processing

In [ ]:
df_test = test_pipeline.fit_transform(test_df)

In [ ]:
df_test

### Modeling

#### Splitting the data

#### Linear Regression

In [ ]:
X = df_train.drop(target,axis=1)
y = df_train[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
class RegressionTransformer:
    """
    Fits a single regression model (or pipeline), performs optional hyperparameter tuning,
    and evaluates standard regression metrics.
    """
    def __init__(self, model, param_grid=None, search_type='grid', cv=5, scoring='r2', n_iter=10, random_state=42):
        """
        model: sklearn estimator or pipeline
        param_grid: dict of hyperparameters to tune
        search_type: 'grid' or 'random'
        cv: number of cross-validation folds
        scoring: metric for selecting best hyperparameters
        n_iter: for RandomizedSearchCV
        """
        self.model = model
        self.param_grid = param_grid
        self.search_type = search_type
        self.cv = cv
        self.scoring = scoring
        self.n_iter = n_iter
        self.random_state = random_state
        self.best_model = None
        self.results = None

    def fit(self, X_train, y_train):
        # If hyperparameter grid is provided, do tuning
        if self.param_grid:
            if self.search_type == 'grid':
                search = GridSearchCV(self.model, self.param_grid, cv=self.cv, scoring=self.scoring, n_jobs=-1)
            elif self.search_type == 'random':
                search = RandomizedSearchCV(self.model, self.param_grid, cv=self.cv, scoring=self.scoring,
                                            n_iter=self.n_iter, n_jobs=-1, random_state=self.random_state)
            else:
                raise ValueError("search_type must be 'grid' or 'random'")
            search.fit(X_train, y_train)
            self.best_model = search.best_estimator_
            self.best_params_ = search.best_params_
        else:
            self.model.fit(X_train, y_train)
            self.best_model = self.model
            self.best_params_ = None
        return self

    def evaluate(self, X_test, y_test):
        if self.best_model is None:
            raise ValueError("Model not fitted yet. Call fit() first.")
        y_pred = self.best_model.predict(X_test)
        metrics = {
            "MSE": mean_squared_error(y_test, y_pred),
            "RMSE": mean_squared_error(y_test, y_pred),
            "MAE": mean_absolute_error(y_test, y_pred),
            "MedianAE": float(median_absolute_error(y_test, y_pred)),
            "R2": r2_score(y_test, y_pred)
        }
        self.results = pd.DataFrame(metrics, index=[0])
        return metrics

    def fit_evaluate(self, X_train, X_test, y_train, y_test):
        self.fit(X_train, y_train)
        return self.evaluate(X_test, y_test)


In [ ]:
# ───────────────────────────────────────────────────────────────
# ✅ 2. New Manager Class to Add & Train Models One-by-One
# ───────────────────────────────────────────────────────────────
class RegressionManager:
    """
    Manages multiple models, allows adding them dynamically,
    trains only the newly added ones, and stores results.
    """
    def __init__(self):
        self.models = {}
        self.trained_models = set()
        self.all_results = []

    def add_model(self, name, model, param_grid=None, search_type='grid',
                  cv=5, scoring='r2', n_iter=10, random_state=42):
        """
        Adds a model but does NOT train it immediately.
        """
        self.models[name] = {
            "model": model,
            "param_grid": param_grid,
            "search_type": search_type,
            "cv": cv,
            "scoring": scoring,
            "n_iter": n_iter,
            "random_state": random_state
        }
        print(f"✅ Model '{name}' added.")

    def train_model(self, name, X_train, X_test, y_train, y_test):
        """
        Trains ONLY the specified model if not already trained.
        """
        if name not in self.models:
            print(f"❌ Model '{name}' not found.")
            return

        if name in self.trained_models:
            print(f"⏭ Model '{name}' already trained. Skipping.")
            return

        cfg = self.models[name]
        print(f"▶ Training model: {name}")

        transformer = RegressionTransformer(
            model=cfg["model"],
            param_grid=cfg["param_grid"],
            search_type=cfg["search_type"],
            cv=cfg["cv"],
            scoring=cfg["scoring"],
            n_iter=cfg["n_iter"],
            random_state=cfg["random_state"]
        )

        metrics = transformer.fit_evaluate(X_train, X_test, y_train, y_test)

        self.all_results.append({
            "Model": name,
            "Best Params": transformer.best_params_,
            **metrics
        })
        self.trained_models.add(name)
        print(f"✅ Training complete for '{name}'.")

    def get_results(self):
        """
        Returns all training results in a DataFrame.
        """
        return pd.DataFrame(self.all_results)


In [ ]:
param_grid = {
    'linereg__fit_intercept': [True, False]  # only if inside a pipeline
}

line_reg = Pipeline([
    ('linereg', LinearRegression())
])
linereg_transformer = RegressionTransformer(model=line_reg, param_grid=param_grid, search_type='grid', cv=5)
linereg_transformer.fit_evaluate(X_train, X_test, y_train, y_test)


#### Stochastic Gradient Regressor

In [ ]:
sgd_pipeline = Pipeline([
  ('sgd', SGDRegressor(max_iter=1000, tol=1e-3, random_state=42))
])

# Hyperparameter grid
sgd_param_grid = {
    'sgd__alpha': [0.0001, 0.001, 0.01],        # regularization
    'sgd__learning_rate': ['constant', 'optimal', 'invscaling'],
    'sgd__eta0': [0.01, 0.1, 0.5]               # initial learning rate
}

# Transformer
sgd_transformer = RegressionTransformer(model=sgd_pipeline,
                                        param_grid=sgd_param_grid,
                                        search_type='grid', cv=5)

# Fit and evaluate
sgd_transformer.fit_evaluate(X_train, X_test, y_train, y_test)

#### Huber Regressor

In [ ]:
huber_pipeline = Pipeline([
    ('huber', HuberRegressor(max_iter=1000))
])

# Hyperparameter grid
huber_param_grid = {
    'huber__epsilon': [1.1, 1.35, 1.5],
    'huber__alpha': [0.0001, 0.001, 0.01]
}

# Transformer
huber_transformer = RegressionTransformer(model=huber_pipeline,
                                          param_grid=huber_param_grid,
                                          search_type='grid', cv=5)

# Fit and evaluate
huber_transformer.fit_evaluate(X_train, X_test, y_train, y_test)

#### Ridge

In [ ]:
ridge_pipeline = Pipeline([
    ('ridge', Ridge())
])

ridge_param_grid = {
    'ridge__alpha': [0.1, 1.0, 10.0],
    'ridge__fit_intercept': [True, False]
}

ridge_transformer = RegressionTransformer(ridge_pipeline, ridge_param_grid, 'grid', cv=5)
ridge_transformer.fit_evaluate(X_train, X_test, y_train, y_test)

In [ ]:
lasso_pipeline = Pipeline([
    ('lasso', Lasso(max_iter=5000))
])

lasso_param_grid = {
    'lasso__alpha': [0.001, 0.01, 0.1, 1.0],
    'lasso__fit_intercept': [True, False]
}

lasso_transformer = RegressionTransformer(lasso_pipeline, lasso_param_grid, 'grid', cv=5)
lasso_transformer.fit_evaluate(X_train, X_test, y_train, y_test)

#### ElasticNet

In [ ]:
en_pipeline = Pipeline([
    ('en', ElasticNet(max_iter=5000))
])
en_param_grid = {
    'en__alpha': [0.001, 0.01, 0.1, 1.0],
    'en__l1_ratio': [0.2, 0.5, 0.8],
    'en__fit_intercept': [True, False]
}

en_transformer = RegressionTransformer(en_pipeline, en_param_grid, 'grid', cv=5)
en_transformer.fit_evaluate(X_train, X_test, y_train, y_test)


#### Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt_pipeline = Pipeline([
    ('dt', DecisionTreeRegressor(random_state=42))
])

dt_param_grid = {
    'dt__max_depth': [None, 5, 10, 20],
    'dt__min_samples_split': [2, 5, 10],
    'dt__min_samples_leaf': [1, 2, 4]
}

dt_transformer = RegressionTransformer(dt_pipeline, dt_param_grid, 'grid', cv=5)
dt_transformer.fit_evaluate(X_train, X_test, y_train, y_test)

#### Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline([
    ('rf', RandomForestRegressor(random_state=42,n_jobs=-1))
])

rf_param_grid = {
    'rf__max_depth': [None, 10, 20],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': [1.0, 'sqrt', 0.8]
}

rf_transformer = RegressionTransformer(rf_pipeline, rf_param_grid, 'grid', cv=5)
rf_transformer.fit_evaluate(X_train, X_test, y_train, y_test)


#### XGBoost Regressor

In [ ]:
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline

xgb_pipeline = Pipeline([
    ('xgb', XGBRegressor(objective='reg:squarederror', random_state=42))
])

xgb_param_grid = {
    # 'xgb__n_estimators': [100, 200,500],
    'xgb__max_depth': [3, 5, 7],
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__subsample': [0.7, 1.0],
    'xgb__colsample_bytree': [0.7, 1.0]
}

xgb_transformer = RegressionTransformer(xgb_pipeline, xgb_param_grid, 'grid', cv=5)
xgb_transformer.fit_evaluate(X_train, X_test, y_train, y_test)
# print("XGBoost Metrics:", xgb_metrics)
# print("Best Params:", xgb_transformer.best_params_)

In [ ]:
xgb_transformer.best_params_

#### AdaBoost Tree Regressor

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# Using a DecisionTree as the base estimator
ada_pipeline = Pipeline([
    ('ada', AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=3), random_state=42))
])

ada_param_grid = {
    'ada__n_estimators': [50, 100, 200],
    'ada__learning_rate': [0.01, 0.05, 0.1, 0.5],
    'ada__estimator__max_depth': [1, 2, 3, 5]
}

ada_transformer = RegressionTransformer(ada_pipeline, ada_param_grid, 'grid', cv=5)
ada_transformer.fit_evaluate(X_train, X_test, y_train, y_test)
# print("AdaBoost Metrics:", ada_metrics)
# print("Best Params:", ada_transformer.best_params_)


#### Support Vectore Regressor

In [ ]:
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore")

In [ ]:
# svr_pipeline = Pipeline([
#     ('svr', LinearSVR(max_iter=100))
# ])

# # Hyperparameter grid
# svr_param_grid = {
#     # 'svr__kernel': ['linear'],
#     'svr__C': [0.1, 1, 10, 100],           # Regularization parameter
#     'svr__epsilon': [0.01, 0.1, 0.2, 0.5], # Tube width
#     # 'svr__degree': [2, 3, 4]               # Only used for 'poly' kernel
# }

# # Transformer
# svr_transformer = RegressionTransformer(svr_pipeline, svr_param_grid, 'grid', cv=5)

# # Fit and evaluate
# svr_transformer.fit_evaluate(X_train, X_test, y_train, y_test)
# # print("SVR Metrics:", svr_metrics)
# # print("Best Params:", svr_transformer.best_params_)

### Full Modeling Pipeline

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error

# ───────────────────────────────────────────────────────────────
# ✅ 1. Base Class: RegressionTransformer (UNCHANGED)
# ───────────────────────────────────────────────────────────────
class RegressionTransformer:
    """
    Fits a single regression model (or pipeline), performs optional hyperparameter tuning,
    and evaluates standard regression metrics.
    """
    def __init__(self, model, param_grid=None, search_type='grid', cv=5, scoring='r2', n_iter=10, random_state=42):
        self.model = model
        self.param_grid = param_grid
        self.search_type = search_type
        self.cv = cv
        self.scoring = scoring
        self.n_iter = n_iter
        self.random_state = random_state
        self.best_model = None
        self.results = None
        self.best_params_ = None

    def fit(self, X_train, y_train):
        if self.param_grid:
            if self.search_type == 'grid':
                search = GridSearchCV(self.model, self.param_grid, cv=self.cv, scoring=self.scoring, n_jobs=-1)
            elif self.search_type == 'random':
                search = RandomizedSearchCV(
                    self.model,
                    self.param_grid,
                    cv=self.cv,
                    scoring=self.scoring,
                    n_iter=self.n_iter,
                    n_jobs=-1,
                    random_state=self.random_state
                )
            else:
                raise ValueError("search_type must be 'grid' or 'random'")
            search.fit(X_train, y_train)
            self.best_model = search.best_estimator_
            self.best_params_ = search.best_params_
        else:
            self.model.fit(X_train, y_train)
            self.best_model = self.model
            self.best_params_ = None
        return self

    def evaluate(self, X_test, y_test):
        if self.best_model is None:
            raise ValueError("Model not fitted yet. Call fit() first.")
        y_pred = self.best_model.predict(X_test)
        metrics = {
            "MSE": mean_squared_error(y_test, y_pred),
            "RMSE": mean_squared_error(y_test, y_pred) ** 0.5,
            "MAE": mean_absolute_error(y_test, y_pred),
            "MedianAE": float(median_absolute_error(y_test, y_pred)),
            "R2": r2_score(y_test, y_pred)
        }
        self.results = pd.DataFrame(metrics, index=[0])
        return metrics

    def fit_evaluate(self, X_train, X_test, y_train, y_test):
        self.fit(X_train, y_train)
        return self.evaluate(X_test, y_test)

# ───────────────────────────────────────────────────────────────
# ✅ 2. New Manager Class to Add & Train Models One-by-One
# ───────────────────────────────────────────────────────────────
class RegressionManager:
    """
    Manages multiple models, allows adding them dynamically,
    trains only the newly added ones, and stores results.
    """
    def __init__(self):
        self.models = {}
        self.trained_models = set()
        self.all_results = []

    def add_model(self, name, model, param_grid=None, search_type='grid',
                  cv=5, scoring='r2', n_iter=10, random_state=42):
        """
        Adds a model but does NOT train it immediately.
        """
        self.models[name] = {
            "model": model,
            "param_grid": param_grid,
            "search_type": search_type,
            "cv": cv,
            "scoring": scoring,
            "n_iter": n_iter,
            "random_state": random_state
        }
        print(f"✅ Model '{name}' added.")

    def train_model(self, name, X_train, X_test, y_train, y_test):
        """
        Trains ONLY the specified model if not already trained.
        """
        if name not in self.models:
            print(f"❌ Model '{name}' not found.")
            return

        if name in self.trained_models:
            print(f"⏭ Model '{name}' already trained. Skipping.")
            return

        cfg = self.models[name]
        print(f"▶ Training model: {name}")

        transformer = RegressionTransformer(
            model=cfg["model"],
            param_grid=cfg["param_grid"],
            search_type=cfg["search_type"],
            cv=cfg["cv"],
            scoring=cfg["scoring"],
            n_iter=cfg["n_iter"],
            random_state=cfg["random_state"]
        )

        metrics = transformer.fit_evaluate(X_train, X_test, y_train, y_test)

        self.all_results.append({
            "Model": name,
            "Best Params": transformer.best_params_,
            "Estimator":transformer.best_model,
            **metrics
        })
        self.trained_models.add(name)
        print(f"✅ Training complete for '{name}'.")

    def get_results(self):
        """
        Returns all training results in a DataFrame.
        """
        return pd.DataFrame(self.all_results)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ──────────────────────────────
# Initialize manager
# ──────────────────────────────
manager = RegressionManager()

# ──────────────────────────────
# 1. Linear Regression
# ──────────────────────────────
lr_pipeline = Pipeline([('linreg', LinearRegression())])
lr_param_grid = {'linreg__fit_intercept': [True, False]}

manager.add_model("LinearRegression", lr_pipeline, lr_param_grid)
manager.train_model("LinearRegression", X_train, X_test, y_train, y_test)

# ──────────────────────────────
# 2. Ridge Regression
# ──────────────────────────────
ridge_pipeline = Pipeline([('ridge', Ridge())])
ridge_param_grid = {'ridge__alpha': [0.1, 1.0, 10.0], 'ridge__fit_intercept': [True, False]}

manager.add_model("Ridge", ridge_pipeline, ridge_param_grid)
manager.train_model("Ridge", X_train, X_test, y_train, y_test)

# ──────────────────────────────
# 3. Lasso Regression
# ──────────────────────────────
lasso_pipeline = Pipeline([('lasso', Lasso(max_iter=5000))])
lasso_param_grid = {'lasso__alpha': [0.001, 0.01, 0.1, 1.0], 'lasso__fit_intercept': [True, False]}

manager.add_model("Lasso", lasso_pipeline, lasso_param_grid)
manager.train_model("Lasso", X_train, X_test, y_train, y_test)

# ──────────────────────────────
# 4. ElasticNet Regression
# ──────────────────────────────
en_pipeline = Pipeline([('en', ElasticNet(max_iter=5000))])
en_param_grid = {'en__alpha': [0.001, 0.01, 0.1, 1.0],
                 'en__l1_ratio': [0.2, 0.5, 0.8],
                 'en__fit_intercept': [True, False]}

manager.add_model("ElasticNet", en_pipeline, en_param_grid)
manager.train_model("ElasticNet", X_train, X_test, y_train, y_test)

# ──────────────────────────────
# 5. SGD Regressor
# ──────────────────────────────
sgd_pipeline = Pipeline([('sgd', SGDRegressor(max_iter=1000, tol=1e-3, random_state=42))])
sgd_param_grid = {'sgd__alpha': [0.0001, 0.001, 0.01],
                  'sgd__learning_rate': ['constant', 'optimal', 'invscaling'],
                  'sgd__eta0': [0.01, 0.1, 0.5]}

manager.add_model("SGDRegressor", sgd_pipeline, sgd_param_grid)
manager.train_model("SGDRegressor", X_train, X_test, y_train, y_test)

# ──────────────────────────────
# 6. Huber Regressor
# ──────────────────────────────
huber_pipeline = Pipeline([('huber', HuberRegressor(max_iter=1000))])
huber_param_grid = {'huber__epsilon': [1.1, 1.35, 1.5],
                    'huber__alpha': [0.0001, 0.001, 0.01]}

manager.add_model("HuberRegressor", huber_pipeline, huber_param_grid)
manager.train_model("HuberRegressor", X_train, X_test, y_train, y_test)

# ──────────────────────────────
# 7. Decision Tree Regressor
# ──────────────────────────────
dt_pipeline = Pipeline([('dt', DecisionTreeRegressor(random_state=42))])
dt_param_grid = {'dt__max_depth': [None, 5, 10, 20],
                 'dt__min_samples_split': [2, 5, 10],
                 'dt__min_samples_leaf': [1, 2, 4]}

manager.add_model("DecisionTree", dt_pipeline, dt_param_grid)
manager.train_model("DecisionTree", X_train, X_test, y_train, y_test)

# ──────────────────────────────
# 8. Random Forest Regressor
# ──────────────────────────────
rf_pipeline = Pipeline([('rf', RandomForestRegressor(random_state=42, n_jobs=-1))])
rf_param_grid = {'rf__max_depth': [None, 10, 20],
                 'rf__min_samples_split': [2, 5],
                 'rf__min_samples_leaf': [1, 2],
                 'rf__max_features': [1.0, 'sqrt', 0.8]}

manager.add_model("RandomForest", rf_pipeline, rf_param_grid)
manager.train_model("RandomForest", X_train, X_test, y_train, y_test)

# ──────────────────────────────
# 9. AdaBoost Regressor
# ──────────────────────────────
ada_pipeline = Pipeline([('ada', AdaBoostRegressor(random_state=42))])
ada_param_grid = {'ada__n_estimators': [50, 100, 200],
                  'ada__learning_rate': [0.01, 0.05, 0.1, 0.5]}

manager.add_model("AdaBoost", ada_pipeline, ada_param_grid)
manager.train_model("AdaBoost", X_train, X_test, y_train, y_test)

# ──────────────────────────────
# 10. XGBoost Regressor
# ──────────────────────────────
xgb_pipeline = Pipeline([('xgb', XGBRegressor(objective='reg:squarederror', random_state=42))])
xgb_param_grid = {'xgb__max_depth': [3, 5, 7],
                  'xgb__learning_rate': [0.01, 0.05, 0.1],
                  'xgb__subsample': [0.7, 1.0],
                  'xgb__colsample_bytree': [0.7, 1.0]}

manager.add_model("XGBoost", xgb_pipeline, xgb_param_grid)
manager.train_model("XGBoost", X_train, X_test, y_train, y_test)

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ──────────────────────────────
# 11. Linear SVR
# ──────────────────────────────
# svr_pipeline = Pipeline([('svr', LinearSVR(random_state=42,max_iter=10000))])
# svr_param_grid = {'svr__C': [0.1, 1, 10, 100],
#                   'svr__epsilon': [0.01, 0.1, 0.2, 0.5]}

# manager.add_model("LinearSVR", svr_pipeline, svr_param_grid)
# manager.train_model("LinearSVR", X_train, X_test, y_train, y_test)


# ──────────────────────────────
# View all results
# ──────────────────────────────
models_df = manager.get_results()
models_df


In [ ]:
models_df = models_df.sort_values(by='R2', ascending=False).reset_index(drop=True)

In [ ]:
models_df

In [ ]:
best_estimator = models_df.at[0,"Estimator"]

In [ ]:
results = best_estimator.predict(df_test)

In [ ]:
results_df = pd.DataFrame({"Id":test_df.Id.values,"SalePrice":np.expm1(results)})

In [ ]:
results_df.to_csv("submission.csv",index=False)

In [ ]:
results_df